In [1]:
import pandas as pd
from tqdm import tqdm

from alphalab.data import HDFData
from alphalab.data import FactorDuckDB

In [2]:
TABLE2FACTOR = {
    "ElementaryFactor": [
        "复权开盘价", "复权最高价", "复权最低价", "复权收盘价", "复权均价",
        "开盘价", "最高价", "最低价", "收盘价", "均价", "复权因子",
        "成交量", "成交金额", "换手率",
        "涨跌停", "涨跌停_一字板",
        "流通市值", "总市值",
        "上市天数", "是否在市", "交易状态",
        "中信行业", "申万行业", "特殊处理", 
    ],
    "IndexConstituentFactor": ["上证50成份权重", "沪深300成份权重", "中证500成份权重", "中证800成份权重", "中证1000成份权重", "中证2000成份权重"],
    "BarraFactor": [
        "Beta", "BookToPrice", "EarningsYield", "Growth", "Leverage", 
        "Liquidity", "Momentum", "NonlinearSize", "ResidualVolatility", "Size"
    ],
}

h5_dir = "//DESKTOP-GC2O6N9/HDF5Data"
h5_dir = "D:/CPResearch/market_data/XYQuantData/HDF5Data"
db_path = "data/stock.duckdb"

In [3]:
h5 = HDFData(f"{h5_dir}/ElementaryFactor/是否在市.hdf5")
s_islisted = h5.fetch_df().stack()
s_universe = s_islisted[s_islisted == 1]
df_universe = s_universe.reset_index().iloc[:, :2]
df_universe.columns = ["date", "code"]

In [ ]:
with FactorDuckDB(db_path) as db:
    db.update_universe(
        df_universe,
        replace=True,
        sync_factor_tables=False,
    )

    for table_name, factor_columns in TABLE2FACTOR.items():
        pbar = tqdm(factor_columns, desc=f"正在处理 {table_name} 表")

        df_fct_list = []
        for factor_name in pbar:
            pbar.set_description(f"正在处理 {table_name}.{factor_name}")

            h5 = HDFData(f"{h5_dir}/{table_name}/{factor_name}.hdf5")

            s_fct = h5.fetch_df().stack()
            s_fct = s_fct.reindex(s_universe.index)
            s_fct.name = factor_name

            df_fct_list.append(s_fct)

        df_table = pd.concat(df_fct_list, axis=1).reset_index()
        df_table = df_table.rename(
            columns={
                "level_0": "date",
                "level_1": "code",
            }
        )

        db.write_factor_table(
            table_name,
            df_table,
            mode="replace_table",
            sync_universe=True,
        )

        print(table_name)
        print(db.table_stats(table_name))
        print(db.universe_match_stats(table_name))

In [4]:
# h5 = HDFData(f"{h5_dir}/ElementaryFactor/是否在市.hdf5")
# df_islisted = h5.fetch_df().stack()
# df_universe = df_islisted[df_islisted == 1]

# for table_name, factor_columns in TABLE2FACTOR.items():
#     pbar = tqdm(factor_columns, desc=f"正在处理 {table_name} 表")
#     df_fct_list = []
#     for factor_name in pbar:
#         pbar.set_description(f"正在处理 {table_name}.{factor_name}")
#         h5 = HDFData(f"{h5_dir}/{table_name}/{factor_name}.hdf5")
#         df_fct = h5.fetch_df().stack().reindex(df_universe.index)
#         df_fct.name = factor_name
#         df_fct_list.append(df_fct)
    
#     df_table = pd.concat(df_fct_list, axis=1).reset_index()
#     df_table = df_table.rename(columns={"level_0": "date", "level_1": "code"})
#     with FactorDuckDB(db_path) as db:
#         db.write_table_frame(table_name, df_table, replace=True)

In [1]:
# with FactorDuckDB(db_path) as db:
#     print(db.list_tables())
#     print(db.list_factors("BarraFactor"))
#     df = db.query(table_name="ElementaryFactor", factors=["复权收盘价"])

In [ ]:
# from pandas.tseries.offsets import DateOffset

# def load_universe(dt):
#     dt0 = (pd.to_datetime(dt) - DateOffset(years=1)).strftime("%Y-%m-%d")
#     sql = f"""
#     WITH 
#     base_codes AS (
#         SELECT code
#         FROM ElementaryFactor
#         WHERE date = '{dt}'
#             AND (特殊处理 IS NULL OR 特殊处理 = '')
#             AND 上市天数 >= 252
#             AND 交易状态 = '交易'
#             AND (涨跌停 IS NULL OR 涨跌停 = 0)
#     ),
#     avg_mcap AS (
#         SELECT code, AVG(总市值) AS avg_market_cap
#         FROM ElementaryFactor
#         WHERE date > '{dt0}' AND date <= '{dt}'
#         GROUP BY code
#     ),
#     mcap_threshold AS (
#         SELECT PERCENTILE_CONT(0.1) WITHIN GROUP (ORDER BY avg_market_cap) AS cap_threshold
#         FROM avg_mcap
#     ),
#     mcap_codes AS (
#         SELECT a.code
#         FROM avg_mcap a, mcap_threshold t
#         WHERE a.avg_market_cap >= t.cap_threshold
#     ),
#     daily_vol AS (
#         SELECT code, 成交金额
#         FROM ElementaryFactor
#         WHERE date > '{dt0}' AND date <= '{dt}'
#     ),
#     avg_vol AS (
#         SELECT code, AVG(成交金额) AS avg_dollar_volume
#         FROM daily_vol
#         GROUP BY code
#     ),
#     vol_threshold AS (
#         SELECT PERCENTILE_CONT(0.1) WITHIN GROUP (ORDER BY avg_dollar_volume) AS vol_threshold
#         FROM avg_vol
#     ),
#     vol_codes AS (
#         SELECT a.code
#         FROM avg_vol a, vol_threshold t
#         WHERE a.avg_dollar_volume >= t.vol_threshold
#     )
#     SELECT code
#     FROM base_codes
#     WHERE code IN (SELECT code FROM mcap_codes)
#         AND code IN (SELECT code FROM vol_codes)
#     """
#     with FactorDuckDB(db_path) as db:
#         res = db.con.execute(sql).df()
#     return res["code"].tolist()